# Lab 02 — Análisis (Equipo 2, Nivel C)

Este notebook **no contiene lógica**: solo importa de `src/`, lee los resultados que generó `python main.py` en `docs/resultados/` y llama a las funciones de `src/plots.py`.

In [ ]:
import sys, json
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
import pandas as pd
from IPython.display import Image, display, Markdown
from src import config, plots
RES = RAIZ / "docs" / "resultados"
FIG = RAIZ / "docs" / "figures"
def leer_json(nombre):
    ruta = RES / nombre
    return json.loads(ruta.read_text(encoding="utf-8")) if ruta.exists() else None
def ver(nombre):
    ruta = FIG / nombre
    if ruta.exists():
        display(Image(filename=str(ruta)))
    else:
        print(f"(figura pendiente: {nombre})")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Datos

In [ ]:
display(pd.read_csv(RES / "auditoria_datos.csv"))
leer_json("splits.json")

## 2. Estrategia (3 indicadores, regla 2 de 3)

In [ ]:
ver("13_correlacion_senales.png")
ver("09_dos_de_tres_vs_un_indicador.png")
pd.read_csv(RES / "un_indicador.csv").query("nivel == 'portafolio'")

## 3. Régimen

In [ ]:
validacion = leer_json("regimenes_validacion.json")
print("Features:", validacion["features_usadas"], "| silhouette:", round(validacion["silhouette_modelo_congelado"], 3))
display(pd.read_csv(RES / "regimen_persistencia_wf_oos.csv"))
for f in ["06a_regimenes_linea_tiempo.png", "06b_features_por_regimen.png", "06c_valor_con_regimenes.png", "06d_transiciones.png"]:
    ver(f)
pd.read_csv(RES / "metricas_por_regimen_wf_oos.csv")

## 4. Portafolio (Risk Parity)

In [ ]:
display(pd.read_csv(RES / "rp_vs_naive_vs_ew.csv")[["nombre", "cagr", "mdd", "calmar", "rc_max", "rc_tsla_nvda"]])
display(pd.read_csv(RES / "estimadores_sigma.csv"))
for f in ["07a_contribuciones_riesgo.png", "07b_senales_s_wf_oos.png", "07c_correlacion_por_regimen.png", "07d_barrido_rebalanceo.png", "08_pesos_estimadores_sigma.png"]:
    ver(f)

## 5. Optimización y walk-forward

In [ ]:
resumen = leer_json("optimizacion_resumen.json")
print(f"Trials totales en TRAIN: {resumen['trials_totales_train']:,}")
print(f"Tiempo de optimización: {resumen['segundos_reloj_optimizacion'] / 3600:.2f} h de reloj")
variantes = pd.read_csv(RES / "comparacion_variantes.csv")
display(variantes[["nombre", "cagr", "mdd", "calmar", "wfe_rendimiento", "wfe_calmar"]])
for _, fila in variantes.iterrows():
    print(f"WFE {fila['nombre']}: rendimiento {fila['wfe_rendimiento']:.2f} | Calmar {fila['wfe_calmar']:.2f}")
for f in ["01_valor_portafolio.png", "02_drawdown.png", "03_rendimientos_wf_oos.png", "12_rolling_vs_anchored.png",
          "14a_optuna_historia.png", "14b_optuna_importancia.png", "14c_optuna_slices.png", "14d_optuna_superficie_3d.png"]:
    ver(f)
pd.read_csv(RES / "metricas_wf_oos.csv")

## 6. Robustez y costos

In [ ]:
print(leer_json("costos_resumen.json"))
display(pd.read_csv(RES / "costos_realistas_train.csv"))
for f in ["04_sensibilidad.png", "05_curva_costos.png", "11_costos_vs_bruto_wf_oos.png", "10_portafolio_vs_individuales_wf_oos.png"]:
    ver(f)

## 7. TEST (se toca una sola vez)

In [ ]:
lock = leer_json("test_lock.json")
if lock is None:
    print("Commit del test: Pendiente")
else:
    print("Commit del test:", lock["commit"], "| fecha:", lock["fecha"])
    display(pd.read_csv(RES / "metricas_test.csv"))
    display(pd.read_csv(RES / "test_secundario_wf.csv"))
    ver("03_rendimientos_test.png")

## 8. Respuestas a las preguntas

Las respuestas completas, con cifras leídas de `docs/resultados/`, están en la sección 13 de `docs/borrador_reporte.md` (generado por `src/report.py`).

In [ ]:
texto = (RAIZ / "docs" / "borrador_reporte.md").read_text(encoding="utf-8")
inicio = texto.find("## 13.")
display(Markdown(texto[inicio:texto.find("## 14.")]))